In [19]:
import math
import cv2
import numpy as np
import gymnasium as gym

from time import sleep
from gymnasium import spaces

from typing import Dict, Any, Sequence, Tuple, Optional


In [20]:
# ---------- Helper functions implementing formulas from the paper ----------
def compute_td_u(B_pu: float, gamma_pu: float) -> float:
    """
    TD,U = 1 / (Bp,u * log2(1 + gamma_pu))
    B_pu: bandwidth (bytes/sec) for DU p to user u (or effective bandwidth)
    gamma_pu: average SNR
    returns average latency (seconds) per byte
    """
    # prevent division by zero or log2(1+gamma)=0
    denom = B_pu * np.log2(1.0 + max(gamma_pu, 1e-12))
    if denom <= 0:
        return np.inf
    return 1.0 / denom

def compute_tm_u(td_u: float, R_M_D: float, U: int) -> float:
    """
    TM,U = TD,U + 1 / (R_M,D / U) = TD,U + U / R_M,D
    R_M_D: total data rate from MEC to DU (bytes/sec)
    U: number of users sharing link equally
    returns average latency (seconds) per byte
    """
    if R_M_D <= 0 or U <= 0:
        return np.inf
    return td_u + (U / R_M_D)

def compute_tc_u(tm_u: float, R_C_M: float, U: int) -> float:
    """
    TC,U = TM,U + 1 / (R_C,M / U) = TM,U + U / R_C,M
    R_C_M: total data rate from Cloud to MEC (bytes/sec)
    """
    if R_C_M <= 0 or U <= 0:
        return np.inf
    return tm_u + (U / R_C_M)

def compute_ttc(rhoT_p: Sequence[float],
                lambda_p: Sequence[float],
                eta: float,
                mu: float) -> float:
    """
    Ttc = (sum_p rhoT_p * lambda_p / eta) / (mu - sum_p lambda_p)
    Conditions: mu - sum(lambda_p) > 0
    rhoT_p, lambda_p arrays must be same length P
    eta: average data size of computation tasks (bytes)
    mu: service rate of MEC server (bytes/sec of processing capacity)
    returns avg computation latency per byte (seconds/byte)
    """
    rhoT_p = np.asarray(rhoT_p, dtype=float)
    lambda_p = np.asarray(lambda_p, dtype=float)
    if eta <= 0:
        raise ValueError("eta must be > 0")
    denom = mu - np.sum(lambda_p)
    if denom <= 0:
        return np.inf
    numerator = np.sum(rhoT_p * lambda_p / eta)
    return numerator / denom

In [21]:
class LatencyModel:
    """
    LatencyModel computes per-tile latency and total latency for requests
    using the formulas from the paper (TD,U, TM,U, TC,U, Ttc and D matrix).
    """
    def __init__(
        self,
        P: int,
        U: int,
        R_M_D: float,
        R_C_M: float,
        mu: float,
        eta: float,
        B_pu_matrix: np.ndarray,
        gamma_pu_matrix: np.ndarray,
        rhoT_p: Sequence[float],
        lambda_p: Sequence[float]
    ):
        self.P = P
        self.U = U
        self.R_M_D = float(R_M_D)
        self.R_C_M = float(R_C_M)
        self.mu = float(mu)
        self.eta = float(eta)

        self.B_pu = np.asarray(B_pu_matrix, dtype=float).reshape(P, U)
        self.gamma_pu = np.asarray(gamma_pu_matrix, dtype=float).reshape(P, U)
        self.rhoT_p = np.asarray(rhoT_p, dtype=float)
        self.lambda_p = np.asarray(lambda_p, dtype=float)

        # precompute TD,U
        self.TD_U = np.zeros((P, U), dtype=float)
        for p in range(P):
            for u in range(U):
                self.TD_U[p, u] = compute_td_u(self.B_pu[p, u], self.gamma_pu[p, u])

    def time_vector(self, p: int, u: int) -> Tuple[float, float, float, float, float]:
        td_u = self.TD_U[p, u]
        tm_u = compute_tm_u(td_u, self.R_M_D, self.U)
        tc_u = compute_tc_u(tm_u, self.R_C_M, self.U)
        ttc = compute_ttc(self.rhoT_p, self.lambda_p, self.eta, self.mu)
        return td_u, tm_u, tc_u, tm_u + ttc, tm_u + ttc

    def tile_latency(self, p: int, u: int, tile_size_bytes: float, events: Dict[str, int]) -> float:
        """
        events dict must contain binary flags for:
         - 'alpha_p_u', 'alpha_M_u', 'alpha_C_u', 'beta_p_u', 'beta_M_u'
        Order corresponds to the 5 columns in the paper's D matrix.
        Returns latency in seconds for that tile (per tile_size_bytes).
        """
        T_vec = np.array(self.time_vector(p, u), dtype=float)
        ES_vec = np.array([
            events.get("alpha_p_u", 0),
            events.get("alpha_M_u", 0),
            events.get("alpha_C_u", 0),
            events.get("beta_p_u", 0),
            events.get("beta_M_u", 0),
        ], dtype=float)
        return float(np.dot(ES_vec, T_vec) * tile_size_bytes)

    def total_request_latency(self, p: int, u: int, tiles: Sequence[Dict[str, Any]]) -> Tuple[float, Sequence[Tuple[str, float]]]:
        tile_latencies = []
        total = 0.0
        for tile in tiles:
            sz = float(tile["size"])
            events = tile.get("events", {})
            tid = tile.get("tile_id", None)
            lat = self.tile_latency(p, u, sz, events)
            tile_latencies.append((tid, lat))
            total += lat
        return total, tile_latencies

In [22]:
class LatencyMixin:
    """
    Mixin that provides compute_latency_for_current_request() and
    an overridable hook to fetch the 'request' structure from the env state.
    """
    
    def __init__(self, *args, latency_model: LatencyModel = None, **kwargs):
        # Mixin expects the concrete env __init__ to call it (use cooperative multiple inheritance)
        super().__init__(*args, **kwargs)
        if latency_model is None:
            raise ValueError("LatencyMixin requires a LatencyModel instance")
        self.latency_model = latency_model

    def fetch_request_for_latency(self) -> Dict[str, Any]:
        """
        Override this in your env if the request is stored differently.
        Expected dictionary format:
          { "p": int, "u": int, "tiles": [{ "tile_id":.., "size":.., "events": {...} }, ...] }
        """
        # default: expect self.last_request attribute (your env should set it before step)
        return getattr(self, "last_request", None)

    def compute_latency_for_current_request(self) -> Dict[str, Any]:
        req = self.fetch_request_for_latency()
        if req is None:
            return {"total_latency": 0.0, "tile_latencies": []}
        p = int(req["p"])
        u = int(req["u"])
        total, tile_latencies = self.latency_model.total_request_latency(p, u, req["tiles"])
        return {"total_latency": total, "tile_latencies": tile_latencies}


In [ ]:
class LatencyWrapper(gym.Env):
    """
    Wraps an existing env and computes latency after each step.
    Use this if you prefer not to subclass your env.
    """

    def __init__(self, env: gym.Env, latency_model: LatencyModel, fetch_request_callable=None):
        super().__init__()
        self.env = env
        self.latency_model = latency_model
        # fetch_request_callable(env, info) -> request dict; default reads env.last_request
        self.fetch_request_callable = fetch_request_callable or (lambda e, info: getattr(e, "last_request", None))
        # mirror spaces
        self.action_space = env.action_space
        self.observation_space = env.observation_space

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        req = self.fetch_request_callable(self.env, info)
        if req is not None:
            p = int(req["p"])
            u = int(req["u"])
            total, tile_latencies = self.latency_model.total_request_latency(p, u, req["tiles"])
            lat_info = {"total_latency": total, "tile_latencies": tile_latencies}
        else:
            lat_info = {"total_latency": 0.0, "tile_latencies": []}

        # choose how to combine: here we subtract latency from reward (common)
        reward = reward - lat_info["total_latency"]
        info = dict(info)  # copy to avoid mutating inner env's info
        info.update(latency=lat_info)
        
        return obs, reward, done, info

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def render(self, mode="human"):
        return self.env.render(mode)

In [24]:
class DummyEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = gym.spaces.Discrete(2)
        self.observation_space = gym.spaces.Box(0.0, 1.0, shape=(1,), dtype=float)
        self.last_request = None

    def step(self, action):
        # produces a fake base reward
        # also populate self.last_request to be used by the latency wrapper
        self.last_request = {
            "p": 0,
            "u": 0,
            "tiles": [
                {"tile_id": "t1", "size": 15000,
                 "events": {"alpha_p_u": 1, "alpha_M_u": 0, "alpha_C_u": 0, "beta_p_u": 0, "beta_M_u": 0}},
                {"tile_id": "t2", "size": 15000,
                 "events": {"alpha_p_u": 0, "alpha_M_u": 1, "alpha_C_u": 0, "beta_p_u": 0, "beta_M_u": 0}}
            ]
        }
        obs = np.array([0.0])
        reward = 1.0  # base reward from env
        done = False
        info = {}
        return obs, reward, done, info

    def reset(self, **kwargs):
        return np.array([0.0]), {}

# Create latency model (replace numbers with your real config)
P = 1; U = 1
lat_model = LatencyModel(
    P=P, 
    U=U,
    R_M_D=5e6, 
    R_C_M=10e6, 
    mu=2e7, 
    eta=2e5,
    B_pu_matrix=np.array([[1e6]]), 
    gamma_pu_matrix=np.array([[5.0]]),
    rhoT_p=[0.2], lambda_p=[0.05]
)

user_env = DummyEnv()
wrapped = LatencyWrapper(user_env, latency_model=lat_model)

obs, reward, done, info = wrapped.step(0)
print("Reward after latency penalty:", reward)
print("Latency info:", info["latency"])


Reward after latency penalty: 0.9853944157829637
Latency info: {'total_latency': 0.014605584217036248, 'tile_latencies': [('t1', 0.0058027921085181235), ('t2', 0.008802792108518124)]}
